# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [4]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog / news', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'portfolio / curriculum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'external partner',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 10 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'projects / products',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects / products', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'training / curriculum',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'partner site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'product page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-9B
Updated
6 days ago
•
868k
•
581
Lightricks/LTX-2.3
Updated
3 days ago
•
175k
•
338
Qwen/Qwen3.5-35B-A3B
Updated
9 days ago
•
1.14M
•
1.03k
Qwen/Qwen3.5-0.8B
Updated
6 days ago
•
406k
•
317
Qwen/Qwen3.5-4B
Updated
6 days ago
•
349k
•
289
Browse 2M+ models
Spaces
Running
on
Zero
Featured
424
Omni Video Factory
🏆
424
text to video, image to video, video extend
Running
on
Zero
144
OBLITERATUS
💥
144
One-click model liberation + chat playground
Running
on
Zero
MCP
1.14k
Wan2.2 14B Preview
🐌
1.14k
generate a video from an image with a text prompt
Running
on
Zero
MCP
Featured
148
FireRed Image Edit 1.0 Fast
🌖
148
FireRed-

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-9B\nUpdated\n6 days ago\n•\n868k\n•\n581\nLightricks/LTX-2.3\nUpdated\n3 days ago\n•\n175k\n•\n338\nQwen/Qwen3.5-35B-A3B\nUpdated\n9 days ago\n•\n1.14M\n•\n1.03k\nQwen/Qwen3.5-0.8B\nUpdated\n6 days ago\n•\n406k\n•\n317\nQwen/Qwen3.5-4B\nUpdated\n6 days ago\n•\n349k\n•\n289\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n424\nOmni Video Factory\n🏆\n424\ntext to video, i

In [26]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [27]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## Who We Are

**Hugging Face** is the vibrant AI community building the future of machine learning (ML). We are the collaboration platform where ML enthusiasts, engineers, scientists, and innovators from around the world come together to create, share, and advance open-source machine learning models, datasets, and applications.

Our mission is to empower the next generation of ML professionals and users by fostering an open, ethical, and collaborative AI ecosystem.

---

## What We Offer

- **Hugging Face Hub:** The central platform for discovering, sharing, and experimenting with over **2 million** machine learning models and **500,000+ datasets**.
- **Spaces:** Collaborative applications running on the platform that allow users to build and showcase ML-powered apps across modalities including text, image, video, audio, and even 3D.
- **Open Source Software Stack:** Accelerate your ML projects with our open-source tools and libraries designed to enable faster experimentation and deployment.
- **Enterprise & Compute Solutions:** Customized enterprise-grade tools and paid compute infrastructure to scale your AI workflows efficiently.
- **Community:** An active, fast-growing global community pushing boundaries in ML and AI, committed to transparency, ethics, and open collaboration.

---

## Our Culture

At Hugging Face, collaboration and openness are at the core of everything we do. We believe that AI should be accessible, ethical, and accelerated through shared knowledge. Our culture embraces:

- **Community-Driven Innovation:** Harnessing collective intelligence for breakthrough ML advancements.
- **Open Source and Transparency:** Delivering tools and models that anyone can use, improve, and build upon.
- **Ethical AI:** Committing to responsible AI development that benefits everyone.
- **Support and Growth:** Empowering machine learning practitioners globally to grow their skills and portfolios by openly sharing work.

---

## Who Uses Hugging Face?

Our platform serves a diverse range of users, including:

- **Machine Learning Engineers & Researchers:** Developing state-of-the-art models and datasets.
- **Startups & Enterprises:** Integrating powerful AI capabilities into their products with our enterprise solutions.
- **Developers & Data Scientists:** Experimenting with cutting-edge technology and accelerating project delivery.
- **Educators and Students:** Learning and teaching ML concepts with open resources.
- **AI Enthusiasts:** Exploring and contributing to a thriving AI ecosystem.

---

## Careers at Hugging Face

Join a mission-driven team where your work will impact the future of AI. At Hugging Face, you will:

- Work alongside leading experts in the AI and ML fields.
- Contribute to open-source projects shaping the AI community.
- Enjoy a culture that values diversity, inclusion, transparency, and continuous learning.
- Take part in a globally connected community with exciting growth opportunities.

Check the Hugging Face website regularly for open roles across engineering, research, product, community, and enterprise functions.

---

## Get Started

Explore, create, and share with Hugging Face today:

- **Browse over 2 million ML models and 500,000+ datasets** on our platform.
- **Build your portfolio** by hosting your projects and applications in Spaces.
- **Join the community** to collaborate and stay updated on AI innovations.
- Access premium enterprise solutions and scalable compute to power your AI needs.

Sign up now at [huggingface.co](https://huggingface.co) and be part of the AI community that’s shaping the future.

---

## Contact & More Information

Visit: [huggingface.co](https://huggingface.co)  
Follow us on social media and join the conversation!

---

*Hugging Face – The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future — a leading collaboration platform for the global machine learning (ML) community. The Hugging Face Hub serves as a central place where anyone — from ML engineers to scientists and end users — can **share, explore, discover, and experiment** with open-source machine learning models, datasets, and applications. By fostering an open and ethical AI future, Hugging Face empowers the next generation of AI innovators worldwide.

---

## What We Offer

### Collaborative Platform

- Host and collaborate on **unlimited public models, datasets, and applications**.
- Access and contribute to **millions of models** — over 2 million models and 500,000+ datasets available.
- Explore diverse AI modalities including **text, image, video, audio, and even 3D**.

### Open Source Stack

- Move faster using our robust open-source tools and libraries that support the entire ML lifecycle.
- Build and share your machine learning profile and portfolio with the community.

### Spaces & Applications

- Discover and run AI applications ("Spaces") built by the community — featuring cutting-edge projects such as text-to-video generation, image editing, and 3D rendering.
- Access over **1 million AI applications** to explore and contribute.

### Paid Compute & Enterprise Solutions

- For teams and businesses, Hugging Face provides paid Compute resources and Enterprise-grade solutions to accelerate machine learning projects with scalable infrastructure and security.

---

## Community & Culture

Hugging Face is a vibrant and fast-growing **AI community** committed to:

- **Open collaboration**: breaking down barriers between individuals and organizations to foster shared progress in AI research and deployment.
- **Ethical AI**: emphasizing responsible use and sharing to contribute to a positive, inclusive AI future.
- **Continuous learning**: supporting users at all levels to develop and showcase their skills in machine learning.
- Inclusion, diversity, and creativity are core values that energize the community spirit.

---

## Customers

Hugging Face serves a wide range of customers including:

- Individual ML enthusiasts, developers, and researchers.
- Academic institutions seeking open resources for AI education and research.
- Startups and large enterprises relying on state-of-the-art ML models and scalable infrastructure.
- AI teams looking to accelerate development with collaborative tools and hosted solutions.

---

## Careers at Hugging Face

Join a passionate team dedicated to pushing the boundaries of machine learning and AI. Opportunities include roles in:

- Machine Learning Engineering & Research
- Software Development
- DevOps & Cloud Infrastructure
- Community Engagement & Developer Relations
- Product Management

Hugging Face offers an innovative, open environment where your work impacts the global AI community. Be part of building the future of AI by empowering millions to collaborate and create.

---

## Get Started Today!

Explore the vast ecosystem of models, datasets, and AI applications at [huggingface.co](https://huggingface.co)

Sign up to:

- Share your own ML projects
- Collaborate with AI experts worldwide
- Accelerate your machine learning workflow

Hugging Face — powering the machine learning community to build a better tomorrow.

---

**Contact & More Info**  
Visit: [https://huggingface.co](https://huggingface.co)  
Explore Docs, Community Forums, and Enterprise Solutions  
Follow us on social media for the latest updates and collaborations  

---

*Hugging Face® is a registered trademark. All rights reserved.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>